# 股票交易策略回测

**策略概要**
- 本金 500 万，固定持有 20 只消费/医药龙头
- 每年 5.1 / 11.1 调仓，权重复平衡至 5%
- 后复权 + 分红入账 + 调仓日再投资

In [ ]:
import config as cfg
from data import fetch_daily_quotes, fetch_dividends, fetch_trade_calendar, fetch_benchmark
from backtest import run_backtest
from metrics import compute_metrics, metrics_report, annual_returns
from plot import plot_nav_and_drawdown, plot_annual_returns, plot_holdings_heatmap

print('模块加载完成')

## Step 1: 获取数据

In [ ]:
print('>>> 获取交易日历...')
trade_dates = fetch_trade_calendar(cfg.START_DATE, cfg.END_DATE)

print(f'\n>>> 获取 {len(cfg.STOCK_CODES)} 只股票日线数据（后复权）...')
quotes = fetch_daily_quotes(cfg.STOCK_CODES, cfg.START_DATE, cfg.END_DATE)

print(f'\n>>> 获取分红送配数据...')
dividends = fetch_dividends(cfg.STOCK_CODES)

print(f'\n>>> 获取基准指数 {cfg.BENCHMARK_INDEX} ...')
benchmark_df = fetch_benchmark(cfg.BENCHMARK_INDEX, cfg.START_DATE, cfg.END_DATE)

print('\n=== 数据获取完成 ===')

## Step 2: 执行回测

In [ ]:
daily, trades, holdings = run_backtest(quotes, dividends, trade_dates, benchmark_df)

## Step 3: 评价指标

In [ ]:
metrics = compute_metrics(daily, trades)
print(metrics_report(metrics))

In [ ]:
# 逐年收益率
annual_df = annual_returns(daily)
annual_df

## Step 4: 可视化

In [ ]:
# 净值曲线 + 回撤曲线
fig_nav, fig_dd = plot_nav_and_drawdown(daily, trades)
fig_nav.show()
fig_dd.show()

In [ ]:
# 年度收益率柱状图
fig_ar = plot_annual_returns(annual_df)
fig_ar.show()

In [ ]:
# 持仓热力图
fig_hm = plot_holdings_heatmap(holdings)
fig_hm.show()

## Step 5: 调仓明细

In [ ]:
# 按调仓日分组查看交易
from collections import defaultdict
trades_by_date = defaultdict(list)
for t in trades:
    trades_by_date[t.date].append(t)

for d, ts in sorted(trades_by_date.items()):
    buy_count = sum(1 for t in ts if t.action == 'BUY')
    sell_count = sum(1 for t in ts if t.action == 'SELL')
    total_amount = sum(t.amount for t in ts)
    total_cost = sum(t.commission + t.stamp_tax + t.transfer_fee for t in ts)
    reason = ts[0].reason
    print(f'{d}  {reason:12s}  买入 {buy_count:2d} 笔 / 卖出 {sell_count:2d} 笔  '
          f'成交额 ¥{total_amount:>12,.0f}  成本 ¥{total_cost:>8,.2f}')